In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('spam.csv', encoding='cp1252')

In [ ]:
df.sample(5)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace=True)

In [ ]:
df.head()

In [ ]:
df.sample(5)

In [ ]:
df.rename(columns={"v1": "target", "v2": "text"}, inplace=True)

In [ ]:
df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
encoder = LabelEncoder()

In [ ]:
df["target"] = encoder.fit_transform(df['target'])

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates(keep='first')

In [ ]:
df.shape

In [ ]:
df['target'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.pie(df['target'].value_counts(), labels=['ham', 'spam'], autopct="%0.2f")
plt.show()

In [ ]:
import nltk

In [ ]:
nltk.download('punkt')

In [ ]:
df['num_char'] = df['text'].apply(len)

In [ ]:
df.head()

In [ ]:
df['num_words'] = df['text'].apply(lambda x: len(nltk.word_tokenize(x)))

In [ ]:
df.head()

In [ ]:
df['num_sent'] = df['text'].apply(lambda x: len(nltk.sent_tokenize(x)))

In [ ]:
df.head()

In [ ]:
df[['num_char', 'num_words', 'num_sent']].describe()

In [ ]:
df[df['target'] == 0][['num_char', 'num_words', 'num_sent']].describe()

In [ ]:
df[df['target'] == 1][['num_char', 'num_words', 'num_sent']].describe()

In [ ]:
import seaborn as sns

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(df[df['target'] == 0]['num_char'])
sns.histplot(df[df['target'] == 1]['num_char'], color='red')

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(df[df['target'] == 0]['num_words'])
sns.histplot(df[df['target'] == 1]['num_words'], color='red')

In [ ]:
plt.figure(figsize=(12,6))
sns.histplot(df[df['target'] == 0]['num_sent'])
sns.histplot(df[df['target'] == 1]['num_sent'], color='red')

In [ ]:
sns.pairplot(df, hue = 'target')

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True)

In [ ]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string
stopwords.words('english')
ps = PorterStemmer()
def transform_txt(text):
    text = text.lower()
    text = nltk.word_tokenize(text)
    y=[]
    for i in text:
        if i.isalnum():
            y.append(i)
    text = y[:]
    y.clear()

    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
    text = y[:]
    y.clear()

    for i in text:
        y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
transform_txt("Who is DD and how can i contact with him can you connect me with him? Because he is a famous singer, and his songs are best in the worlds, whenever he is singing a song i sings his songs as a pronoun.")

In [ ]:
df['transformed_text'] = df['text'].apply(transform_txt)

In [ ]:
df.head()

In [ ]:
from wordcloud import WordCloud
wc = WordCloud(width=500, height=500, min_font_size=10, background_color='white')

In [ ]:
spam_wc = wc.generate(df[df['target'] == 1]['transformed_text'].str.cat(sep=" "))
ham_wc = wc.generate(df[df['target'] == 0]['transformed_text'].str.cat(sep=" "))

In [ ]:
plt.figure(figsize=(15, 6))
plt.imshow(spam_wc)

In [ ]:
plt.figure(figsize=(15, 6))
plt.imshow(ham_wc)

In [ ]:
 df.head()

In [ ]:
spam_corpus = []
for msg in df[df['target'] == 1]['transformed_text'].tolist():
    for word in msg.split():
        spam_corpus.append(word)

In [ ]:
len(spam_corpus)

In [ ]:
from collections import Counter
sns.barplot(x = pd.DataFrame(Counter(spam_corpus).most_common(30))[0], y = pd.DataFrame(Counter(spam_corpus).most_common(30))[1])
plt.xticks(rotation = "vertical")
plt.show()

In [ ]:
ham_corpus = []
for msg in df[df['target'] == 0]['transformed_text'].tolist():
    for word in msg.split():
        ham_corpus.append(word)

In [ ]:
len(ham_corpus)

In [ ]:
from collections import Counter
sns.barplot(x = pd.DataFrame(Counter(ham_corpus).most_common(30))[0], y = pd.DataFrame(Counter(ham_corpus).most_common(30))[1])
plt.xticks(rotation = "vertical")
plt.show()

In [ ]:
df.head()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
cv = CountVectorizer()
tfidf = TfidfVectorizer()

In [ ]:
X = tfidf.fit_transform(df['transformed_text']).toarray()

In [ ]:
X.shape

In [ ]:
y = df['target'].values

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)


In [ ]:
X_train.shape, X_test.shape

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,classification_report

In [ ]:
mnb = MultinomialNB()
mnb.fit(X_train,y_train)


In [ ]:
y_pred = mnb.predict(X_test)

In [ ]:
mnb = MultinomialNB()
mnb.fit(X_train,y_train)

In [ ]:
y_pred = mnb.predict(X_test)

In [ ]:
print(accuracy_score(y_test,y_pred))
print(precision_score(y_test,y_pred))

In [ ]:
confusion_matrix(y_test,y_pred)

In [ ]:
sns.heatmap(confusion_matrix(y_test,y_pred), annot=True, fmt='d')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.show()

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
def predict_msg(msg):
    t = transform_txt(msg)
    v = tfidf.transform([t]).toarray()
    r = mnb.predict(v)[0]
    if r == 1:
        return "Spam"
    else:
        return "Ham"

In [ ]:
predict_msg("Congratulations you have won rs.100000 prize pool")

In [ ]:
predict_msg("do you have free passes of rangtaali garba night")

In [ ]:
from dotenv import load_dotenv
from google import genai
import os
load_dotenv()
client = genai.Client()

In [ ]:
 def ai_predict(message):
     prompt = "Classify this message as spam or ham. Reply only one word like spam or ham: " + message
     response = client.models.generate_content(model = "gemini-3.5-flash-lite", contents=prompt)
     answer = response.text.strip().lower()
     if "spam" in answer:
        return 1
     else:
         return 0

In [ ]:
ai_predict("Sorry, I'll call later")

In [ ]:
import time
sample = df.sample(20, random_state=2)
ai_preds = []
for msg in sample['text']:
    ai_preds.append(ai_predict(msg))
    time.sleep(5)

In [ ]:
print(accuracy_score(sample['target'], ai_preds))

In [ ]:
print(precision_score(sample['target'], ai_preds))

In [ ]:
print(confusion_matrix(sample['target'], ai_preds))

In [ ]:
ai_predict("ALERT: Your bank account has been temporarily locked due to suspicious activity. Log in immediately to verify your identity: findmyphotos.ai")

In [ ]:
import time
sample = df.sample(20, random_state=2)
ai_preds = []
for msg in sample['text']:
    ai_preds.append(ai_predict(msg))
    time.sleep(5)

In [ ]:
print(accuracy_score(sample['target'], ai_preds))

In [ ]:
print(precision_score(sample['target'], ai_preds))

In [ ]:
print(confusion_matrix(sample['target'], ai_preds))